In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

INPUT_FILES = {
    "VN30": RAW_DIR / "investing_vn30_daily_raw.csv",
    "VNMidcap": RAW_DIR / "investing_vnmidcap_daily_raw.csv",
    "VNSmallcap": RAW_DIR / "investing_vnsmallcap_daily_raw.csv",
    "VNAllshare": RAW_DIR / "investing_vnallshare_daily_raw.csv",
}

COLUMN_MAP = {
    "Ngày": "date",
    "Lần cuối": "close",
    "Mở": "open",
    "Cao": "high",
    "Thấp": "low",
    "KL": "volume",
    "% Thay đổi": "change_pct",
}

def parse_price(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().replace(",", "")
    if x in ["", "-", "nan", "None"]:
        return np.nan
    return float(x)

def parse_volume(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().replace(",", "")
    if x in ["", "-", "nan", "None"]:
        return np.nan

    multiplier = 1.0

    if x[-1].upper() == "K":
        multiplier = 1_000
        x = x[:-1]
    elif x[-1].upper() == "M":
        multiplier = 1_000_000
        x = x[:-1]
    elif x[-1].upper() == "B":
        multiplier = 1_000_000_000
        x = x[:-1]

    return float(x) * multiplier

def parse_change_pct(x):
    if pd.isna(x):
        return np.nan

    x = str(x).strip().replace("%", "").replace(",", ".")
    if x in ["", "-", "nan", "None"]:
        return np.nan

    return float(x) / 100

def clean_investing_file(path, index_name):
    df = pd.read_csv(path, encoding="utf-8-sig")

    missing_cols = [c for c in COLUMN_MAP if c not in df.columns]
    if missing_cols:
        raise ValueError(f"{index_name}: missing columns {missing_cols}")

    df = df.rename(columns=COLUMN_MAP)

    df["date"] = pd.to_datetime(df["date"], dayfirst=True, errors="coerce")

    for col in ["close", "open", "high", "low"]:
        df[col] = df[col].apply(parse_price)

    df["volume"] = df["volume"].apply(parse_volume)
    df["change_pct"] = df["change_pct"].apply(parse_change_pct)
    df["index"] = index_name

    df = df[["date", "index", "close", "open", "high", "low", "volume", "change_pct"]]
    df = df.sort_values("date").drop_duplicates(subset=["date"]).reset_index(drop=True)

    # Basic return from close price
    df["simple_return"] = df["close"].pct_change()
    df["log_return"] = 100 * np.log(df["close"] / df["close"].shift(1))

    # Data-quality flags
    df["ohlc_valid"] = (
        (df["high"] >= df[["open", "close", "low"]].max(axis=1)) &
        (df["low"] <= df[["open", "close", "high"]].min(axis=1))
    )

    df["change_pct_diff"] = (df["change_pct"] - df["simple_return"]).abs()
    df["change_pct_valid"] = df["change_pct_diff"].isna() | (df["change_pct_diff"] <= 0.001)

    return df

cleaned_data = {}
quality_rows = []

for index_name, path in INPUT_FILES.items():
    df = clean_investing_file(path, index_name)
    cleaned_data[index_name] = df

    output_path = PROCESSED_DIR / f"investing_{index_name.lower()}_daily_clean.csv"
    output_path = Path(str(output_path).replace("vnmidcap", "vnmidcap")
                                      .replace("vnsmallcap", "vnsmallcap")
                                      .replace("vnallshare", "vnallshare"))
    df.to_csv(output_path, index=False)

    quality_rows.append({
        "index": index_name,
        "rows": len(df),
        "start_date": df["date"].min().date(),
        "end_date": df["date"].max().date(),
        "missing_values": int(df.isna().sum().sum()),
        "duplicate_dates": int(df["date"].duplicated().sum()),
        "weekend_dates": int(df["date"].dt.weekday.ge(5).sum()),
        "ohlc_invalid_rows": int((~df["ohlc_valid"]).sum()),
        "change_pct_invalid_rows": int((~df["change_pct_valid"]).sum()),
        "extreme_abs_log_return_gt_10pct": int((df["log_return"].abs() > 10).sum()),
    })

quality_report = pd.DataFrame(quality_rows)
quality_report.to_csv(PROCESSED_DIR / "data_quality_report.csv", index=False)

display(quality_report)

In [ ]:
all_cleaned = pd.concat(cleaned_data.values(), ignore_index=True)

close_wide = (
    all_cleaned
    .pivot(index="date", columns="index", values="close")
    .sort_index()
)

print("Full close-price matrix:")
print(close_wide.info())
print(close_wide.notna().sum())

close_common = close_wide.dropna()

print("\nCommon sample:")
print(close_common.index.min(), "→", close_common.index.max())
print("Number of price observations:", len(close_common))

close_output = PROCESSED_DIR / "vietnam_size_indices_close_common_20141120_20251231.csv"
close_common.to_csv(close_output)

display(close_common.head())
display(close_common.tail())

In [ ]:
returns = 100 * np.log(close_common / close_common.shift(1))
returns = returns.dropna()

returns_output = PROCESSED_DIR / "vietnam_size_indices_log_returns_common_20141121_20251231.csv"
returns.to_csv(returns_output)

print("Return sample:")
print(returns.index.min(), "→", returns.index.max())
print("Number of return observations:", len(returns))

display(returns.describe(percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]).T)

print("\nExtreme return count | abs(log_return) > 10%:")
print((returns.abs() > 10).sum())

display(returns.head())
display(returns.tail())

In [ ]:
import matplotlib.pyplot as plt

for col in returns.columns:
    plt.figure(figsize=(12, 4))
    plt.plot(returns.index, returns[col])
    plt.title(f"{col} daily log returns")
    plt.xlabel("Date")
    plt.ylabel("Log return (%)")
    plt.tight_layout()
    plt.show()

In [ ]:
# ============================================================
# MISSING VALUE CHECKS FOR DATASETS USED LATER
# ============================================================

from pathlib import Path
import pandas as pd

PROCESSED_DIR = Path("data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

def missing_value_report(df, dataset_name):
    """
    Create a missing-value report for one dataframe.
    """
    report = pd.DataFrame({
        "dataset": dataset_name,
        "column": df.columns,
        "missing_count": df.isna().sum().values,
        "missing_pct": (df.isna().mean().values * 100).round(4),
        "non_missing_count": df.notna().sum().values,
        "dtype": df.dtypes.astype(str).values
    })

    return report.sort_values(["missing_count", "missing_pct"], ascending=False)


# ------------------------------------------------------------
# 1. Check missing values in each cleaned raw index file
# ------------------------------------------------------------

cleaned_missing_reports = []

for index_name, df in cleaned_data.items():
    report = missing_value_report(df, f"{index_name}_cleaned")
    cleaned_missing_reports.append(report)

    print(f"\nMissing values: {index_name}_cleaned")
    display(report)

cleaned_missing_report_all = pd.concat(cleaned_missing_reports, ignore_index=True)

cleaned_missing_report_all.to_csv(
    PROCESSED_DIR / "missing_values_cleaned_index_files.csv",
    index=False
)


# ------------------------------------------------------------
# 2. Check missing values in wide close-price dataset before dropna
# ------------------------------------------------------------

close_wide_missing_report = missing_value_report(
    close_wide.reset_index(),
    "close_wide_before_dropna"
)

print("\nMissing values: close_wide_before_dropna")
display(close_wide_missing_report)

close_wide_missing_report.to_csv(
    PROCESSED_DIR / "missing_values_close_wide_before_dropna.csv",
    index=False
)


# ------------------------------------------------------------
# 3. Check missing values in common close-price dataset
# This is the price dataset used to compute returns
# ------------------------------------------------------------

close_common_missing_report = missing_value_report(
    close_common.reset_index(),
    "close_common_after_dropna"
)

print("\nMissing values: close_common_after_dropna")
display(close_common_missing_report)

close_common_missing_report.to_csv(
    PROCESSED_DIR / "missing_values_close_common.csv",
    index=False
)


# ------------------------------------------------------------
# 4. Check missing values in final log-return dataset
# This is the main dataset for EDA, GARCH and VaR
# ------------------------------------------------------------

returns_missing_report = missing_value_report(
    returns.reset_index(),
    "log_returns_common"
)

print("\nMissing values: log_returns_common")
display(returns_missing_report)

returns_missing_report.to_csv(
    PROCESSED_DIR / "missing_values_log_returns_common.csv",
    index=False
)


# ------------------------------------------------------------
# 5. Final pass/fail summary
# ------------------------------------------------------------

missing_summary = pd.DataFrame([
    {
        "dataset": "close_wide_before_dropna",
        "rows": len(close_wide),
        "columns": close_wide.shape[1],
        "total_missing": int(close_wide.isna().sum().sum()),
        "status": "CHECK_ONLY"
    },
    {
        "dataset": "close_common_after_dropna",
        "rows": len(close_common),
        "columns": close_common.shape[1],
        "total_missing": int(close_common.isna().sum().sum()),
        "status": "PASS" if close_common.isna().sum().sum() == 0 else "FAIL"
    },
    {
        "dataset": "log_returns_common",
        "rows": len(returns),
        "columns": returns.shape[1],
        "total_missing": int(returns.isna().sum().sum()),
        "status": "PASS" if returns.isna().sum().sum() == 0 else "FAIL"
    }
])

print("\nFinal missing-value summary")
display(missing_summary)

missing_summary.to_csv(
    PROCESSED_DIR / "missing_values_final_summary.csv",
    index=False
)